# LeetCode 2441 — Largest Positive Integer That Exists With Its Negative bằng Python

Link đề: https://leetcode.com/problems/largest-positive-integer-that-exists-with-its-negative/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Two Pointers, Sorting

Skeleton LeetCode:

```python
class Solution:
    def findMaxK(self, nums: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu bài toán: tìm **số nguyên dương `k` lớn nhất** sao cho **cả `k` và `-k`** đều có mặt trong mảng; không có thì trả `-1`.
2. Dùng `set` để kiểm tra "`-k` có trong mảng không" trong `O(1)`, và hiểu tại sao dùng `set` tốt hơn quét lại mảng.
3. Biết kỹ thuật **hai con trỏ** trên mảng đã sắp xếp: một con trỏ ở đầu (âm nhất), một ở cuối (dương nhất).
4. Viết 4 cách: vét cạn mọi cặp, `set`, sắp xếp + hai con trỏ, và giao của hai tập (dương, âm đã đổi dấu); đối chiếu bằng test ngẫu nhiên.
5. Tránh các lỗi: lấy `max(nums)`, khởi tạo `best = 0` thay vì `-1`, kiểm tra `abs(x) in set` (tự khớp với chính nó), và chỉ kiểm tra theo một chiều khi duyệt.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` **không chứa số `0`**. Hãy tìm **số nguyên dương `k` lớn nhất** sao cho **`-k` cũng có trong `nums`**. Trả về `k`. Nếu không có số nào như vậy, trả về `-1`.

| Ví dụ | nums | Output | Giải thích |
|---|---|---|---|
| 1 | `[-1,2,-3,3]` | `3` | `3` và `-3` đều có mặt; `-1` có mặt nhưng `1` thì không nên không tính; chỉ `3` thỏa |
| 2 | `[-1,10,6,7,-7,1]` | `7` | `1` (`-1`, `1`) và `7` (`-7`, `7`) đều thỏa; lớn nhất `7` |
| 3 | `[-10,8,6,7,-2,-3]` | `-1` | không có cặp `k`, `-k` |

Constraints: `1 <= nums.length <= 1000`; `-1000 <= nums[i] <= 1000`; `nums[i] != 0`.

In [1]:
nums = [-1, 10, 6, 7, -7, 1]
pool = set(nums)
for x in sorted(pool):
    if x > 0:
        print(f"k={x}: có {-x} trong mảng? {-x in pool}")

k=1: có -1 trong mảng? True
k=6: có -6 trong mảng? False
k=7: có -7 trong mảng? True
k=10: có -10 trong mảng? False


## 3. Kiến thức nền

### 3.1. Số đối và dấu

Số đối của `x` là `-x`. Số dương `k` "tồn tại cùng số đối" nếu cả `k` và `-k` đều xuất hiện. Vì `nums[i] != 0` nên mọi số hoặc dương hoặc âm (nếu có `0`, nó là số đối của chính nó và cần xử lý riêng).

In [2]:
for x in (3, -3, 7, -7):
    print(x, -x, abs(x), x > 0)

3 -3 3 True
-3 3 3 False
7 -7 7 True
-7 7 7 False


### 3.2. `set`: tập hợp và kiểm tra thuộc

`set(nums)` bỏ trùng và cho `x in s` trung bình `O(1)`. Còn `x in list` mất `O(n)`. Khi cần hỏi "có phần tử này không?" nhiều lần, hãy dựng `set`.

In [3]:
nums = [-1, 2, -3, 3]
s = set(nums)
print(s)
print(3 in s, -3 in s, 1 in s, -1 in s)

{2, 3, -3, -1}
True True False True


### 3.3. `max` với giá trị mặc định

`max(iterable, default=-1)` trả `-1` nếu `iterable` rỗng, thay vì ném lỗi. Rất hợp với "không có thì trả `-1`".

In [4]:
print(max([x for x in [3, 7] if x > 5], default=-1))
print(max([x for x in [3, 4] if x > 5], default=-1))

7
-1


### 3.4. Sắp xếp và hai con trỏ

Mảng đã sắp tăng dần: `a[0]` âm nhất, `a[-1]` dương nhất. Đặt `i = 0`, `j = len(a) - 1`. So sánh tổng `a[i] + a[j]`:

- `= 0`: tìm thấy cặp đối nhau `(a[i], a[j])`.
- `< 0`: số âm quá "to" (độ lớn lớn hơn `a[j]`), tăng `i`.
- `> 0`: số dương quá lớn, giảm `j`.

In [5]:
a = sorted([-10, 8, 6, 7, -2, -3])
print(a)
i, j = 0, len(a) - 1
while i < j:
    print(f"i={i} ({a[i]}), j={j} ({a[j]}), tổng={a[i] + a[j]}")
    if a[i] + a[j] == 0:
        print("tìm thấy", a[j]); break
    if a[i] + a[j] < 0:
        i += 1
    else:
        j -= 1

[-10, -3, -2, 6, 7, 8]
i=0 (-10), j=5 (8), tổng=-2
i=1 (-3), j=5 (8), tổng=5
i=1 (-3), j=4 (7), tổng=4
i=1 (-3), j=3 (6), tổng=3
i=1 (-3), j=2 (-2), tổng=-5


### 3.5. Phép toán tập hợp

Giao hai tập: `A & B` (hoặc `A.intersection(B)`). Nếu `pos = {x for x in nums if x > 0}` và `neg = {-x for x in nums if x < 0}` (đã đổi dấu), thì `pos & neg` là tập mọi `k` thỏa đề.

In [6]:
nums = [-1, 10, 6, 7, -7, 1]
pos = {x for x in nums if x > 0}
neg = {-x for x in nums if x < 0}
print(pos, neg, pos & neg, max(pos & neg, default=-1))

{1, 10, 6, 7} {1, 7} {1, 7} 7


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Làm tay: mỗi số dương `k` trong mảng, ta hỏi "có `-k` không?". Chọn `k` lớn nhất trong những `k` trả lời "có".

Ví dụ 2: `[-1, 10, 6, 7, -7, 1]`.

| k dương | có `-k`? |
|---|---|
| 1 | `-1` có ⇒ **thỏa** |
| 6 | `-6` không có |
| 7 | `-7` có ⇒ **thỏa** |
| 10 | `-10` không có |

Hai ứng viên: `1` và `7`; lớn nhất `7`.

Ví dụ 3: `[-10, 8, 6, 7, -2, -3]`: các dương `8, 6, 7` đều không có đối ⇒ `-1`.

Lưu ý: nếu `-k` có mặt mà `k` không có thì **không** tính (ví dụ `-3` đứng một mình). Ta cần **cả hai**.

In [7]:
nums = [-1, 10, 6, 7, -7, 1]
answer = -1
for x in nums:
    if x > 0 and -x in nums:
        print("ứng viên:", x)
        answer = max(answer, x)
print("đáp án:", answer)

ứng viên: 7
ứng viên: 1
đáp án: 7


## 5. Cách 1 — Brute Force / baseline: vét cạn mọi cặp

**Ý tưởng:** duyệt mọi cặp `(i, j)`; nếu `nums[i] + nums[j] == 0` thì `nums[i]` và `nums[j]` đối nhau, ứng viên là số dương trong hai số, tức `abs(nums[i])`. Lấy giá trị lớn nhất.

In [8]:
from typing import List

class SolutionBrute:
    def findMaxK(self, nums: List[int]) -> int:
        best = -1
        n = len(nums)
        for i in range(n):
            for j in range(i + 1, n):
                if nums[i] + nums[j] == 0:
                    best = max(best, abs(nums[i]))
        return best

In [9]:
sol = SolutionBrute()
print(sol.findMaxK([-1, 2, -3, 3]))             # 3
print(sol.findMaxK([-1, 10, 6, 7, -7, 1]))      # 7
print(sol.findMaxK([-10, 8, 6, 7, -2, -3]))     # -1

3
7
-1


**Độ phức tạp:** Time `O(n^2)` (~500.000 cặp với `n = 1000`), Space `O(1)`. **Nút thắt:** với mỗi số ta quét lại phần còn lại của mảng để tìm số đối; một `set` trả lời câu hỏi đó trong `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** biến câu hỏi "có số đối không?" thành tra cứu `set`.

- **Invariant:** `s = set(nums)` chứa toàn bộ giá trị; `best` là `k` lớn nhất đã thấy trong các phần tử đã duyệt.
- **Quy tắc:** với mỗi `x` trong `nums`: **nếu** `x > 0` **và** `-x in s` **thì** `best = max(best, x)`.
- Chỉ xét `x > 0` để mỗi cặp chỉ tính một lần và để ứng viên luôn dương. Không cần xét `x < 0` vì nếu `-x` dương có mặt thì chính nó sẽ được xét khi duyệt đến số dương đó.
- Đây là "dựng `set` trước, duyệt sau", không phải "vừa duyệt vừa thêm" (cách đó cần kiểm tra cả hai chiều).

## 7. Các cách tiếp theo

### Cách 2 — `set` toàn mảng, duyệt các số dương

In [10]:
from typing import List

class SolutionSet:
    def findMaxK(self, nums: List[int]) -> int:
        s = set(nums)
        best = -1
        for x in s:
            if x > 0 and -x in s:
                best = max(best, x)
        return best

In [11]:
print(SolutionSet().findMaxK([-1, 2, -3, 3]))             # 3
print(SolutionSet().findMaxK([-1, 10, 6, 7, -7, 1]))      # 7
print(SolutionSet().findMaxK([-10, 8, 6, 7, -2, -3]))     # -1

3
7
-1


**Độ phức tạp:** Time `O(n)` trung bình, Space `O(n)`.

### Cách 3 — Sắp xếp rồi hai con trỏ

Sắp xếp tăng dần, rồi đặt `i` ở đầu, `j` ở cuối. Tổng `a[i] + a[j]`: bằng `0` ⇒ trả `a[j]` (vì `j` là dương **lớn nhất** còn lại nên là đáp án ngay); `< 0` ⇒ `i += 1`; `> 0` ⇒ `j -= 1`.

In [12]:
from typing import List

class SolutionTwoPointers:
    def findMaxK(self, nums: List[int]) -> int:
        a = sorted(nums)
        i, j = 0, len(a) - 1
        while i < j:
            total = a[i] + a[j]
            if total == 0:
                return a[j]
            if total < 0:
                i += 1
            else:
                j -= 1
        return -1

In [13]:
print(SolutionTwoPointers().findMaxK([-1, 2, -3, 3]))             # 3
print(SolutionTwoPointers().findMaxK([-1, 10, 6, 7, -7, 1]))      # 7
print(SolutionTwoPointers().findMaxK([-10, 8, 6, 7, -2, -3]))     # -1

3
7
-1


**Độ phức tạp:** Time `O(n log n)` (sắp xếp, hai con trỏ chỉ `O(n)`), Space `O(n)` cho bản sao đã sắp xếp (hoặc `O(1)` nếu sắp xếp tại chỗ).

### Cách 4 — Giao của hai tập

Dựng tập các số dương và tập đối của các số âm; giao của chúng là mọi `k` hợp lệ.

In [14]:
from typing import List

class SolutionIntersect:
    def findMaxK(self, nums: List[int]) -> int:
        pos = {x for x in nums if x > 0}
        neg = {-x for x in nums if x < 0}
        return max(pos & neg, default=-1)

In [15]:
print(SolutionIntersect().findMaxK([-1, 2, -3, 3]))             # 3
print(SolutionIntersect().findMaxK([-1, 10, 6, 7, -7, 1]))      # 7
print(SolutionIntersect().findMaxK([-10, 8, 6, 7, -2, -3]))     # -1

3
7
-1


**Độ phức tạp:** Time `O(n)` trung bình, Space `O(n)`.

## 8. TRACE từng bước

Cách hai con trỏ trên `[-10, 8, 6, 7, -2, -3]` → sau sắp xếp `[-10, -3, -2, 6, 7, 8]`:

| Bước | `i` | `j` | `a[i]` | `a[j]` | tổng | Hành động |
|---|---|---|---|---|---|---|
| 1 | 0 | 5 | -10 | 8 | -2 | `< 0` ⇒ `i += 1` |
| 2 | 1 | 5 | -3 | 8 | 5 | `> 0` ⇒ `j -= 1` |
| 3 | 1 | 4 | -3 | 7 | 4 | `> 0` ⇒ `j -= 1` |
| 4 | 1 | 3 | -3 | 6 | 3 | `> 0` ⇒ `j -= 1` |
| 5 | 1 | 2 | -3 | -2 | -5 | `< 0` ⇒ `i += 1` |
| 6 | 2 | 2 | | | | `i == j`, dừng ⇒ `-1` |

In [16]:
def trace_two_pointers(nums):
    a = sorted(nums)
    print("đã sắp xếp:", a)
    i, j = 0, len(a) - 1
    step = 0
    while i < j:
        total = a[i] + a[j]
        step += 1
        print(f"bước {step}: i={i}({a[i]}), j={j}({a[j]}), tổng={total}", end=" -> ")
        if total == 0:
            print(f"tìm thấy đáp án {a[j]}")
            return a[j]
        if total < 0:
            print("i += 1")
            i += 1
        else:
            print("j -= 1")
            j -= 1
    print("con trỏ gặp nhau => -1")
    return -1

print(trace_two_pointers([-10, 8, 6, 7, -2, -3]))
print(trace_two_pointers([-1, 10, 6, 7, -7, 1]))

đã sắp xếp: [-10, -3, -2, 6, 7, 8]
bước 1: i=0(-10), j=5(8), tổng=-2 -> i += 1
bước 2: i=1(-3), j=5(8), tổng=5 -> j -= 1
bước 3: i=1(-3), j=4(7), tổng=4 -> j -= 1
bước 4: i=1(-3), j=3(6), tổng=3 -> j -= 1
bước 5: i=1(-3), j=2(-2), tổng=-5 -> i += 1
con trỏ gặp nhau => -1
-1
đã sắp xếp: [-7, -1, 1, 6, 7, 10]
bước 1: i=0(-7), j=5(10), tổng=3 -> j -= 1
bước 2: i=0(-7), j=4(7), tổng=0 -> tìm thấy đáp án 7
7


## 9. Code nộp LeetCode

Dùng `set`:

In [17]:
from typing import List

class Solution:
    def findMaxK(self, nums: List[int]) -> int:
        s = set(nums)
        best = -1
        for x in s:
            if x > 0 and -x in s:
                best = max(best, x)
        return best

In [18]:
print(Solution().findMaxK([-1, 2, -3, 3]))
print(Solution().findMaxK([-1, 10, 6, 7, -7, 1]))
print(Solution().findMaxK([-10, 8, 6, 7, -2, -3]))

3
7
-1


## 10. Vì sao thuật toán đúng

1. **Đặc trưng đáp án.** `k > 0` là hợp lệ ⇔ `k in s` và `-k in s`. Đáp án là hợp lệ lớn nhất, hoặc `-1` nếu không có.
2. **Cách `set`.** Duyệt các giá trị của `s`; với `x > 0`, kiểm tra `-x in s`. Điều kiện `x in s` đã đúng vì `x` lấy từ `s`. Vậy `best` là cực đại các `k` hợp lệ.
3. **Hai con trỏ.** Hai con trỏ giữ bất biến: *mọi dương lớn hơn `a[j]` đã bị loại (không có đối trong phần còn lại) và mọi âm bé hơn `a[i]` đã bị loại*.
   - Nếu `a[i] + a[j] > 0` thì `a[j] > |a[i]|`: `a[i]` là âm "mạnh nhất" còn lại nên không phần tử âm nào còn lại có đối `a[j]` (đối của `a[j]` sẽ là `-a[j] < a[i]`, đã bị loại hoặc không có). Loại `a[j]`.
   - Nếu `a[i] + a[j] < 0` thì `|a[i]| > a[j]`: `a[j]` là dương lớn nhất còn lại nên không dương nào còn lại có đối `a[i]`. Loại `a[i]`.
   - Nếu `= 0`: `a[j]` là dương lớn nhất còn lại có đối, mà các dương lớn hơn đã bị loại, nên `a[j]` là đáp án.
4. **Giao hai tập.** `pos & neg` chính là tập các `k` thỏa cả `k in nums` và `-k in nums`.
5. **Vét cạn.** Mọi cặp `(i, j)` có tổng `0` được xét; ứng viên `abs(nums[i])` được cập nhật nên không bỏ sót.

## 11. Phân tích Time / Space Complexity

Đặt `n = len(nums)`.

| Cách | Time | Space |
|---|---|---|
| Vét cạn mọi cặp | `O(n^2)` | `O(1)` |
| `set` | `O(n)` trung bình | `O(n)` |
| Sắp xếp + hai con trỏ | `O(n log n)` | `O(n)` (hoặc `O(1)` sắp xếp tại chỗ) |
| Giao hai tập | `O(n)` trung bình | `O(n)` |

Với `n <= 1000` mọi cách đều nhanh. Nếu muốn không dùng bộ nhớ phụ, hai con trỏ (sắp xếp tại chỗ) là lựa chọn tốt.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

In [19]:
sols = [SolutionBrute(), SolutionSet(), SolutionTwoPointers(), SolutionIntersect(), Solution()]
cases = [
    ([-1, 2, -3, 3], 3),                  # ví dụ 1
    ([-1, 10, 6, 7, -7, 1], 7),           # ví dụ 2
    ([-10, 8, 6, 7, -2, -3], -1),         # ví dụ 3
    ([1], -1),                            # một phần tử dương
    ([-1], -1),                           # một phần tử âm
    ([1, -1], 1),                         # cặp nhỏ nhất
    ([-1, 1], 1),                         # cặp thứ tự ngược
    ([3, -3], 3),                         # dương đứng trước (bắt lỗi duyệt một chiều)
    ([5, 5, -5, -5], 5),                  # trùng lặp
    ([1000, -1000], 1000),                # biên lớn nhất
    ([-1000, -999, 999], 999),            # chỉ 999 có đối? -999 và 999
    ([1, 2, 3], -1),                      # toàn dương
    ([-1, -2, -3], -1),                   # toàn âm
    ([2, -2, 3, -4, 4], 4),               # nhiều cặp -> lớn nhất
    ([7, 7, 7], -1),                      # lặp mà không có đối
]
for nums, want in cases:
    for s in sols:
        assert s.findMaxK(nums) == want, (type(s).__name__, nums, s.findMaxK(nums), want)
print("tất cả test cố định đều đúng với cả 5 cách:", len(cases))

tất cả test cố định đều đúng với cả 5 cách: 15


### 12.2. Đối chiếu ngẫu nhiên (có seed)

Sinh đúng constraints: `1 <= n <= 1000`, `nums[i] in [-1000, 1000]` và **khác 0**.

In [20]:
import random
random.seed(2441)

def rand_nonzero(lo, hi):
    while True:
        x = random.randint(lo, hi)
        if x != 0:
            return x

def oracle(nums):
    s = set(nums)
    cands = [k for k in range(1, 1001) if k in s and -k in s]
    return cands[-1] if cands else -1

hits = 0
for _ in range(1500):
    n = random.randint(1, 12)
    bound = random.choice([3, 5, 10, 1000])
    nums = [rand_nonzero(-bound, bound) for _ in range(n)]
    want = oracle(nums)
    for s in sols:
        assert s.findMaxK(nums) == want, (type(s).__name__, nums)
    hits += (want != -1)
print("1500 test nhỏ khớp oracle; có đáp án khác -1 trong", hits, "test")
assert 100 < hits < 1500

for _ in range(80):
    n = random.randint(500, 1000)
    nums = [rand_nonzero(-1000, 1000) for _ in range(n)]
    want = oracle(nums)
    for s in sols:
        assert s.findMaxK(nums) == want
print("80 test lớn khớp oracle")

1500 test nhỏ khớp oracle; có đáp án khác -1 trong 721 test


80 test lớn khớp oracle


### 12.3. Test có cặp được cấy và test không có cặp

Cấy cặp `(k, -k)` vào mảng toàn dương để chắc chắn đáp án `>= k`; và dựng mảng không thể có cặp (chỉ dương, hoặc chỉ âm, hoặc không có số đối) để chắc chắn đáp án `-1`.

In [21]:
random.seed(31)
for _ in range(200):
    n = random.randint(2, 50)
    nums = [random.randint(1, 1000) for _ in range(n)]
    k = random.randint(1, 1000)
    nums[random.randrange(n)] = k
    nums[random.randrange(n)] = -k
    if k not in nums or -k not in nums:       # hai vị trí ngẫu nhiên có thể đè nhau
        continue
    ans = Solution().findMaxK(nums)
    assert ans >= k and ans == oracle(nums)

for _ in range(200):
    n = random.randint(1, 50)
    nums = [random.randint(1, 1000) for _ in range(n)]
    assert Solution().findMaxK(nums) == -1
    assert Solution().findMaxK([-x for x in nums]) == -1
print("cấy cặp và các mảng cùng dấu: OK")

cấy cặp và các mảng cùng dấu: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Trả về `max(nums)` hoặc số lớn nhất mà không kiểm tra số đối

In [22]:
from typing import List

def sai_lay_max(nums: List[int]) -> int:
    return max(nums)                                        # SAI: không kiểm tra -x

nums = [-10, 8, 6, 7, -2, -3]
print("đúng:", Solution().findMaxK(nums), "| sai:", sai_lay_max(nums))
assert Solution().findMaxK(nums) != sai_lay_max(nums)

đúng: -1 | sai: 8


**Vì sao sai:** số lớn nhất `8` không có đối `-8`. Đề chỉ chấp nhận `k` khi **cả `k` và `-k`** tồn tại; không có thì trả `-1`.

### Lỗi 2: Khởi tạo `best = 0` (hoặc quên trả `-1`)

In [23]:
def sai_best_0(nums: List[int]) -> int:
    s = set(nums)
    best = 0                                                # SAI: phải là -1
    for x in s:
        if x > 0 and -x in s:
            best = max(best, x)
    return best

nums = [1, 2, 3]
print("đúng:", Solution().findMaxK(nums), "| sai:", sai_best_0(nums))
assert Solution().findMaxK(nums) != sai_best_0(nums)

đúng: -1 | sai: 0


**Vì sao sai:** khi không có cặp nào, đề yêu cầu `-1`; `0` không phải đáp án hợp lệ (và `0` bị cấm trong `nums`, nên không thể lẫn với kết quả thật).

### Lỗi 3: Kiểm tra `abs(x) in set` (tự khớp với chính mình)

In [24]:
def sai_abs(nums: List[int]) -> int:
    s = set(nums)
    best = -1
    for x in s:
        if abs(x) in s:                                     # SAI: x luôn tự khớp khi x > 0 (và x < 0 thì abs(x) cần có thật)
            best = max(best, abs(x))
    return best

nums = [-10, 8, 6, 7, -2, -3]
print("đúng:", Solution().findMaxK(nums), "| sai:", sai_abs(nums))
assert Solution().findMaxK(nums) != sai_abs(nums)

đúng: -1 | sai: 8


**Vì sao sai:** với `x = 8`, `abs(8) = 8` luôn nằm trong `s` (chính nó). Ta cần `-x` (số **đối**), không phải `abs(x)`.

### Lỗi 4: Duyệt một chiều và chỉ kiểm tra khi số âm đến sau

In [25]:
def sai_mot_chieu(nums: List[int]) -> int:
    seen = set()
    best = -1
    for x in nums:
        if x < 0 and -x in seen:                            # SAI: chỉ bắt được trường hợp dương đứng trước
            best = max(best, -x)
        seen.add(x)
    return best

nums = [-3, 3]
print("đúng:", Solution().findMaxK(nums), "| sai:", sai_mot_chieu(nums))
assert Solution().findMaxK(nums) != sai_mot_chieu(nums)

đúng: 3 | sai: -1


**Vì sao sai:** khi duyệt từng phần tử và chỉ xét `x < 0`, cặp mà số âm đứng **trước** số dương bị bỏ sót. Có hai cách an toàn: dựng `set` toàn mảng trước, hoặc kiểm tra `-x in seen` cho **mọi** `x` (cả dương lẫn âm).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tìm phần tử tương ứng/đối/bù của mỗi phần tử" ⇒ `set` / `dict` để tra cứu `O(1)` (giống Two Sum).
- "Không có thì trả `-1`" ⇒ khởi tạo `-1` hoặc dùng `max(..., default=-1)`.
- Ràng buộc `nums[i] != 0` giúp bỏ qua trường hợp `0` là số đối của chính nó; nếu đề cho phép `0`, hãy xử lý riêng.
- Duyệt chỉ số dương `x > 0` giúp mỗi cặp chỉ đếm một lần.
- Hai con trỏ trên mảng sắp xếp rất hợp với "hai số có tổng bằng 0" (tương tự 3Sum, Two Sum II).
- Test nhanh bằng `[3, -3]` và `[-3, 3]` để kiểm tra không phụ thuộc thứ tự.

## 15. Biến thể của bài

1. **Cho phép `0`**: `0` có đối là chính nó; cần ít nhất một số `0` (trả `0`) khi không có `k > 0`.
2. **Trả về tất cả `k` hợp lệ** (sắp xếp giảm dần).
3. **Đếm số cặp** `(i, j)` có `nums[i] + nums[j] == 0`.
4. **Hai số có tổng bằng `target`** (Two Sum): `set` / dict + `target - x`.
5. **Số nguyên nhỏ nhất `k`** thay vì lớn nhất.
6. **Cặp có độ chênh lệch cho trước** thay vì tổng bằng `0`: `x + d in s`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | thử mọi cặp có tổng `0` | `O(n^2)` | `O(1)` | `n` rất nhỏ, đối chứng |
| 2. `set` | dựng `set`, kiểm tra `-x` cho `x > 0` | `O(n)` | `O(n)` | cách chuẩn, ngắn gọn |
| 3. Sắp xếp + hai con trỏ | thu hẹp từ hai đầu | `O(n log n)` | `O(1)`–`O(n)` | khi hạn chế bộ nhớ |
| 4. Giao hai tập | `pos & neg` | `O(n)` | `O(n)` | rất dễ đọc, một dòng |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** `Solution().findMaxK([4, -4, 2, -3, 3])` là gì?

**Bài 2 (tính tay).** Với `nums = [-5, 1, 5, -1, 2]`, liệt kê các `k` hợp lệ và chọn đáp án.

**Bài 3 (điền TODO).** Hoàn thành hàm `co_cap_doi(s, k)` kiểm tra `k` và `-k` cùng thuộc tập `s`.

In [26]:
def co_cap_doi(s, k):
    # TODO: trả về True nếu k > 0 và cả k lẫn -k đều thuộc s
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `tat_ca_k(nums)` trả về **danh sách tất cả** `k` hợp lệ, sắp xếp giảm dần.

**Bài 5 (biến thể / mini challenge).** Cho phép `nums` chứa `0`. Viết `findMaxK_with_zero(nums)`: trả về `k >= 0` lớn nhất sao cho `k` và `-k` đều có mặt (với `k = 0` chỉ cần có số `0` trong mảng); nếu không có trả `-1`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: cặp `(4, -4)` và `(3, -3)`; lớn nhất `4`. Bài 2: kiểm tra `1`, `2`, `5`.
- **Mức 2 (vừa):** Bài 3: `k > 0 and k in s and -k in s`. Bài 4: `sorted((x for x in s if x > 0 and -x in s), reverse=True)`.
- **Mức 3 (sâu):** Bài 5: tìm `k` lớn nhất `> 0` như bài chính; nếu không có, `0 in s` thì trả `0`, ngược lại `-1`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [27]:
# Bài 1
assert Solution().findMaxK([4, -4, 2, -3, 3]) == 4

# Bài 2: 1 (có -1) ; 5 (có -5) ; 2 không có -2 -> {1, 5} -> 5
assert Solution().findMaxK([-5, 1, 5, -1, 2]) == 5

# Bài 3
def co_cap_doi(s, k):
    return k > 0 and k in s and -k in s

# Bài 4
def tat_ca_k(nums):
    s = set(nums)
    return sorted((x for x in s if x > 0 and -x in s), reverse=True)

# Bài 5
def findMaxK_with_zero(nums):
    s = set(nums)
    best = max((x for x in s if x > 0 and -x in s), default=-1)
    if best == -1 and 0 in s:
        return 0
    return best

assert co_cap_doi({1, -1, 2}, 1) is True
assert co_cap_doi({1, -1, 2}, 2) is False
assert co_cap_doi({0}, 0) is False
assert tat_ca_k([-5, 1, 5, -1, 2]) == [5, 1]
assert tat_ca_k([1, 2, 3]) == []
assert tat_ca_k([-3, 3, 3, -3]) == [3]
assert findMaxK_with_zero([0, 1]) == 0
assert findMaxK_with_zero([0, 3, -3]) == 3
assert findMaxK_with_zero([1, 2]) == -1
assert findMaxK_with_zero([0]) == 0
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (dựng `set`, kiểm tra số đối)

```python
class Solution:
    def findMaxK(self, nums: List[int]) -> int:
        s = set(nums)
        best = -1
        for x in s:
            if x > 0 and -x in s:
                best = max(best, x)
        return best
```

**Checklist**

- [ ] Kiểm tra `-x in s` (số đối), **không** phải `abs(x) in s`?
- [ ] Chỉ xét `x > 0` để ứng viên luôn dương?
- [ ] Khởi tạo `best = -1` (không phải `0`)?
- [ ] `set` được dựng **trước** khi duyệt (không phụ thuộc thứ tự)?
- [ ] Đã thử `[3, -3]`, `[-3, 3]`, mảng toàn dương/toàn âm?
- [ ] Hai con trỏ (nếu dùng): đã sắp xếp trước và điều kiện dừng `i < j`?
- [ ] Signature đúng `findMaxK(self, nums: List[int]) -> int` và có `from typing import List`?